# 04 · Оценка Student Model

Блокнот сравнивает исходную **Qwen2.5-0.5B-Instruct** и LoRA-адаптеры
на 200 задачах из общего `test.jsonl`. Для теста настройки оценки уже зафиксированы.

1. Откройте этот `.ipynb` в Google Colab и выберите **Среда выполнения → Сменить среду выполнения → GPU**.
2. Запускайте ячейки сверху вниз. В ячейке с данными загрузите командный `project_data.zip` или оригинальный `test.jsonl`.
3. По умолчанию проверяются baseline и опубликованный B42. Для проверки другого запуска добавьте путь к папке `adapter/` в `ADAPTERS`. Для короткой технической проверки можно выбрать `validation` и загрузить архив или `validation.jsonl`.

Оценка использует только условие задачи и общую инструкцию. Правильный ответ нужен после генерации.
Жадное декодирование, лимит 192 новых токена, шаблон Qwen ChatML. В последней непустой строке должно быть `#### число`.
Отсутствующий или неоднозначный финальный ответ входит в знаменатель accuracy как ошибка. [Командный протокол](https://github.com/Gionicerr/PLACE_HOLDER/blob/fadc462d7efc2e9a9c8bac6ea47bbe83b48379b4/protocol.py)

In [ ]:
%pip -q install transformers==4.53.3 peft==0.16.0 accelerate==1.8.1 huggingface-hub==0.34.4 safetensors==0.5.3

## Окружение и командный код

Используется GPU, если он доступен. Блокнот использует предустановленную сборку PyTorch в Colab.
Точный коммит нужен для шаблона запроса, парсера, проверки хеша данных и загрузки B42.

In [ ]:
import subprocess
import sys
from pathlib import Path

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(torch.__version__, torch.cuda.get_device_name() if DEVICE == "cuda" else "CPU")

PIN = "fadc462d7efc2e9a9c8bac6ea47bbe83b48379b4"
PROJECT = Path.cwd()
if not all((PROJECT / name).is_file() for name in (
    "protocol.py", "configs/A123.json", "results/B42/adapter/adapter_config.json"
)):
    PROJECT = Path.cwd() / f"team-project-{PIN[:7]}"
    if not (PROJECT / ".git").is_dir():
        subprocess.run(["git", "init", "-q", str(PROJECT)], check=True)
    if not (PROJECT / "protocol.py").is_file():
        subprocess.run(["git", "fetch", "-q", "--depth", "1",
                        "https://github.com/Gionicerr/PLACE_HOLDER.git", PIN], cwd=PROJECT, check=True)
        subprocess.run(["git", "checkout", "-q", "--detach", "FETCH_HEAD"], cwd=PROJECT, check=True)
    head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT, text=True).strip()
    if head != PIN:
        raise RuntimeError(f"Expected repository revision {PIN}, got {head}")
sys.path.insert(0, str(PROJECT))

## Данные и адаптеры

Для финального прогона оставьте `SPLIT = "test"` и `BATCH_SIZE = 8`.
`ADAPTERS = []` запускает только baseline. Адаптер из распакованного пакета участника № 4 подхватится автоматически.
Для адаптера из Google Drive можно смонтировать диск и указать полный путь до каталога с `adapter_config.json` и `adapter_model.safetensors`.

In [ ]:
SPLIT = "test"
BATCH_SIZE = 8
DATA_PATH = Path("project_data.zip")
ADAPTERS = [("B42", PROJECT / "results/B42/adapter")]
personal_adapter = Path("personal/B123/adapter")
if personal_adapter.is_dir():
    ADAPTERS.append(("B123", personal_adapter))

if not DATA_PATH.exists():
    for candidate in (Path(f"{SPLIT}.jsonl"), PROJECT / "data/project_data.zip"):
        if candidate.is_file():
            DATA_PATH = candidate
            break
if not DATA_PATH.exists() and "google.colab" in sys.modules:
    from google.colab import files

    uploaded = files.upload()
    candidates = [Path(name) for name in uploaded if Path(name).suffix.lower() == ".zip"
                  or Path(name).name == f"{SPLIT}.jsonl"]
    if len(candidates) == 1:
        DATA_PATH = candidates[0]
if not DATA_PATH.exists():
    raise FileNotFoundError("Укажите путь к project_data.zip или исходному файлу разбиения.")

## Общая проверка ответов

Для всех моделей используется один парсер из `protocol.py`. Он сравнивает числа точно,
поэтому `1,200.00` и `1200` считаются равными. Финальная строка с единицами измерения или
несколько строк с `####` считаются ошибкой формата.

In [ ]:
"""Evaluate the team's original Student and optional LoRA adapters on the fixed split."""

import argparse
import csv
import hashlib
import io
import re
import statistics
import time
from contextlib import nullcontext
from importlib.metadata import version
from pathlib import Path

from protocol import (CONFIG, DATA, ROOT, atomic_text, chat_prompt, read_json,
                      read_split, save_json, save_jsonl, score, user_prompt)

PLAN = read_json(ROOT / "configs" / "A123.json")
MODEL, REVISION = PLAN["student"], PLAN["revision"]
MAX_NEW_TOKENS = CONFIG["generation"]["max_new_tokens"]


def adapter_arg(text):
    label, _, path = text.partition("=")
    if not re.fullmatch(r"[A-Za-z][A-Za-z0-9_-]*", label) or label.lower() == "baseline" or not path:
        raise argparse.ArgumentTypeError("expected LABEL=adapter_directory (baseline is reserved)")
    return label, Path(path)


def adapter_metadata(path):
    config = read_json(path / "adapter_config.json")
    weights = path / "adapter_model.safetensors"
    if (config.get("peft_type"), config.get("base_model_name_or_path"), config.get("revision")) != (
        "LORA", MODEL, REVISION
    ):
        raise ValueError(f"{path} must be a LoRA adapter for {MODEL} at {REVISION}")
    with weights.open("rb") as stream:
        metadata = {"adapter_sha256": hashlib.file_digest(stream, "sha256").hexdigest()}
    run_config, run_results = path.parent / "run_config.json", path.parent / "run_results.json"
    if run_config.is_file():
        metadata["seed"] = read_json(run_config).get("seed")
    if run_results.is_file():
        metadata["train_time_seconds"] = read_json(run_results).get("train_time_seconds")
    return metadata


def evaluate_model(model, tokenizer, questions, label, batch_size, context):
    import torch
    from transformers import GenerationConfig

    eos_token_id = model.generation_config.eos_token_id
    eos_ids = frozenset(eos_token_id if isinstance(eos_token_id, list) else [eos_token_id])
    config = GenerationConfig(
        do_sample=False, num_beams=1, max_new_tokens=MAX_NEW_TOKENS,
        pad_token_id=tokenizer.pad_token_id, eos_token_id=eos_token_id,
    )
    predictions = []
    started = time.perf_counter()
    with torch.inference_mode(), context:
        for start in range(0, len(questions), batch_size):
            batch = questions[start:start + batch_size]
            inputs = tokenizer([chat_prompt(row["question"]) for row in batch], padding=True,
                               add_special_tokens=False, return_tensors="pt").to(model.device)
            output = model.generate(**inputs, generation_config=config)
            generated = output[:, inputs.input_ids.shape[1]:].tolist()
            for row, token_ids in zip(batch, generated, strict=True):
                length = next((i for i, token in enumerate(token_ids) if token in eos_ids), len(token_ids))
                response = tokenizer.decode(token_ids[:length], skip_special_tokens=True).strip()
                extracted, correct = score(response, row["reference_answer"])
                predictions.append({
                    "prompt_id": row["id"], "reference_answer": row["reference_answer"],
                    "response": response, "extracted_answer": extracted, "is_correct": correct,
                    "response_tokens": length, "hit_token_limit": length == MAX_NEW_TOKENS,
                })
            if len(predictions) == len(questions) or (start // batch_size + 1) % 5 == 0:
                print(f"{label}: {len(predictions)}/{len(questions)}", flush=True)
    return predictions, time.perf_counter() - started


def summarize(rows):
    n = len(rows)
    correct = sum(row["is_correct"] for row in rows)
    invalid = sum(row["extracted_answer"] is None for row in rows)
    lengths = [row["response_tokens"] for row in rows]
    return {
        "n": n, "correct": correct, "accuracy": correct / n,
        "wrong_number": n - correct - invalid, "invalid": invalid, "invalid_format_rate": invalid / n,
        "token_limit_count": sum(row["hit_token_limit"] for row in rows),
        "response_tokens_mean": statistics.mean(lengths),
        "response_tokens_median": statistics.median(lengths), "response_tokens_max": max(lengths),
    }


def run(data, output, adapters=(), split="test", batch_size=8, device="auto"):
    import torch
    from filelock import FileLock
    from transformers import AutoModelForCausalLM, AutoTokenizer

    if batch_size < 1:
        raise ValueError("batch_size must be positive")
    if split not in ("test", "validation"):
        raise ValueError("split must be test or validation")
    adapters = list(adapters)
    if len({label.lower() for label, _ in adapters}) != len(adapters):
        raise ValueError("Adapter labels must be distinct")
    adapter_info = {label: adapter_metadata(path) for label, path in adapters}
    stage = "pilot" if split == "validation" else "full"
    if data.name == f"{split}.jsonl" and data.is_file():
        data = data.parent
    questions = read_split(data, split, stage)
    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA is unavailable in this environment")
    if device == "cpu":
        torch.set_num_threads(4)
    torch.set_float32_matmul_precision("highest")

    output.mkdir(parents=True, exist_ok=True)
    with FileLock(str(output / ".evaluate.lock"), timeout=0):
        if any(path.name != ".evaluate.lock" for path in output.iterdir()):
            raise ValueError(f"Use a new or empty output directory: {output}")
        tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION, padding_side="left")
        native = tokenizer.apply_chat_template([{"role": "user", "content": user_prompt("Template check")}],
                                               tokenize=False, add_generation_prompt=True)
        if native != chat_prompt("Template check"):
            raise ValueError("The tokenizer chat template differs from the team's protocol")
        model = AutoModelForCausalLM.from_pretrained(
            MODEL, revision=REVISION, use_safetensors=True, torch_dtype=torch.float32,
        ).to(device).eval()
        if adapters:
            from peft import PeftModel

            first_label, first_adapter = adapters[0]
            model = PeftModel.from_pretrained(model, first_adapter, adapter_name=first_label).eval()
            for label, path in adapters[1:]:
                model.load_adapter(path, adapter_name=label, is_trainable=False)

        metrics = {}
        for label in ["baseline", *adapter_info]:
            context = nullcontext()
            if adapters:
                if label == "baseline":
                    context = model.disable_adapter()
                else:
                    model.set_adapter(label)
            rows, elapsed = evaluate_model(model, tokenizer, questions, label, batch_size, context)
            save_jsonl(output / f"{label}.predictions.jsonl", rows)
            metrics[label] = {**summarize(rows), **adapter_info.get(label, {}),
                              "inference_seconds": round(elapsed, 3)}
            metrics[label]["delta_pp"] = round(100 * (metrics[label]["accuracy"] - metrics["baseline"]["accuracy"]), 8)
            result = metrics[label]
            print(f"{label}: {result['correct']}/{result['n']} = {result['accuracy']:.1%}, "
                  f"invalid: {result['invalid']}, delta: {result['delta_pp']:+.1f} pp", flush=True)

        manifest = {
            "model": MODEL, "revision": REVISION, "split": split, "stage": stage,
            "source_split_sha256": CONFIG["data"][split]["sha256"],
            "prompt_system": CONFIG["generation"]["system"], "prompt_instruction": CONFIG["generation"]["instruction"],
            "decoding": "greedy", "max_new_tokens": MAX_NEW_TOKENS, "batch_size": batch_size,
            "dtype": "float32", "device": device, "gpu": torch.cuda.get_device_name() if device == "cuda" else None,
            "versions": {name: version(name) for name in ("torch", "transformers", "peft", "accelerate")},
        }
        fields = ["model", "split", "stage", "seed", "n", "correct", "accuracy", "delta_pp",
                  "wrong_number", "invalid", "invalid_format_rate", "token_limit_count",
                  "response_tokens_mean", "response_tokens_median", "response_tokens_max",
                  "train_time_seconds", "inference_seconds"]
        stream = io.StringIO()
        writer = csv.DictWriter(stream, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        for label, result in metrics.items():
            writer.writerow({"model": label, "split": split, "stage": stage, **result})
        atomic_text(output / "metrics.csv", stream.getvalue())
        report = {"protocol": manifest, "models": metrics}
        save_json(output / "metrics.json", report)
    return report

In [ ]:
assert score("6 + 4 = 10\n#### 10", "10") == ("10", True)
assert score("#### +1,200.00", "1200") == ("1200.00", True)
assert score("#### -.50", "-0.5") == ("-0.50", True)
assert score("#### 10 kg", "10") == (None, False)
assert score("#### 1\n#### 10", "10") == (None, False)
assert score("5 + 5 = 10", "10") == (None, False)

## Запуск и выгрузка

В `metrics.csv` поле `accuracy` — доля от 0 до 1, `delta_pp` — изменение относительно baseline
в процентных пунктах. `metrics.json` также содержит параметры запуска. Файлы `*.predictions.jsonl`
сохраняют текст и оценку каждой задачи для разбора ошибок.

In [ ]:
from tempfile import mkdtemp
from zipfile import ZIP_DEFLATED, ZipFile

output_dir = Path(mkdtemp(prefix=f"evaluation-{SPLIT}-", dir=Path.cwd()))
report = run(DATA_PATH, output_dir, ADAPTERS, SPLIT, BATCH_SIZE, DEVICE)
result_archive = output_dir.with_suffix(".zip")
with ZipFile(result_archive, "w", compression=ZIP_DEFLATED) as archive:
    for path in sorted(output_dir.glob("*.json*")) + [output_dir / "metrics.csv"]:
        archive.write(path, path.name)
print("Результаты:", result_archive)
# В Colab можно скачать ZIP из панели «Файлы».

## Результаты контрольного запуска

Оценка выполнена на NVIDIA GeForce RTX 5060, PyTorch 2.7.1+cu128, float32, батч 8. Все модели прошли один и тот же `test.jsonl`.

| Модель | Верно | Accuracy | Δ к baseline | Ошибки формата |
| :--- | ---: | ---: | ---: | ---: |
| baseline | 0/200 | 0.0% | +0.0 п. п. | 199 |
| B42 | 74/200 | 37.0% | +37.0 п. п. | 73 |
| B123 | 68/200 | 34.0% | +34.0 п. п. | 76 |